In [1]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")

[INFO] torch/torchvision versions not as required, installing nightly versions.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu113
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.met

torch version: 2.3.1+cu121
torchvision version: 0.18.1+cu121


In [1]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    import data_setup, engine
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    import data_setup, engine

[INFO] Couldn't find torchinfo... installing it.


In [2]:
import os
!pip install opendatasets
import opendatasets as od

In [3]:
od.download('https://www.kaggle.com/datasets/ahedjneed/fancy-watche-images')

Dataset URL: https://www.kaggle.com/datasets/ahedjneed/fancy-watche-images


100%|██████████| 63.4M/63.4M [00:02<00:00, 29.9MB/s]


In [4]:
import requests
import zipfile
from pathlib import Path
os.listdir('fancy-watche-images/content/images')

['Breguet',
 'Audemars Piguet',
 'Vacheron Constantin',
 'Maurice Lacroix',
 'Rolex',
 'Rado',
 'watches.csv',
 'Omega',
 'Breitling',
 'Tissot',
 'Patek Philippe']

In [5]:

import shutil
from sklearn.model_selection import train_test_split

In [6]:
base_dir = 'fancy-watche-images/content/images'
train_dir = 'fancy-watche-images/content/train'
test_dir = 'fancy-watche-images/content/test'

In [7]:
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

In [8]:
watch_brands = os.listdir(base_dir)

In [9]:
for brand in os.listdir(base_dir):
    brand_path = os.path.join(base_dir, brand)


    if os.path.isdir(brand_path):
        images = os.listdir(brand_path)

        if len(images) > 1:
            train_images, test_images = train_test_split(images, test_size=0.2, random_state=42)

            train_brand_dir = os.path.join(train_dir, brand)
            test_brand_dir = os.path.join(test_dir, brand)
            os.makedirs(train_brand_dir, exist_ok=True)
            os.makedirs(test_brand_dir, exist_ok=True)


            for img in train_images:
                shutil.move(os.path.join(brand_path, img), os.path.join(train_brand_dir, img))

            for img in test_images:
                shutil.move(os.path.join(brand_path, img), os.path.join(test_brand_dir, img))
        else:
            print(f"Not enough images in folder {brand} to split.")


In [10]:
from pathlib import Path


base_dir = Path('fancy-watche-images/content/images')


for path in base_dir.rglob('*'):
    print(path)


fancy-watche-images/content/images/Breguet
fancy-watche-images/content/images/Audemars Piguet
fancy-watche-images/content/images/Vacheron Constantin
fancy-watche-images/content/images/Maurice Lacroix
fancy-watche-images/content/images/Rolex
fancy-watche-images/content/images/Rado
fancy-watche-images/content/images/watches.csv
fancy-watche-images/content/images/Omega
fancy-watche-images/content/images/Breitling
fancy-watche-images/content/images/Tissot
fancy-watche-images/content/images/Patek Philippe


In [11]:
from pathlib import Path

test_dir = Path('fancy-watche-images/content/test')

# Print files in the known folder
for path in test_dir.glob('*.jpg'):
    print(path)

In [12]:
normalize = transforms.Normalize(mean=[0.485,0.456,406],
                                 std=[0.229,0.224,0.225])

In [13]:

manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [14]:

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=manual_transforms, # resize, convert images to between 0 & 1 and normalize them
                                                                               batch_size=32) # set mini-batch size to 32

train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x7b482832bee0>,
 ['Audemars Piguet',
  'Breguet',
  'Breitling',
  'Maurice Lacroix',
  'Omega',
  'Patek Philippe',
  'Rado',
  'Rolex',
  'Tissot',
  'Vacheron Constantin'])

In [15]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

In [16]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # .DEFAULT = best available weights from pretraining on ImageNet
weights

EfficientNet_B0_Weights.IMAGENET1K_V1

In [17]:
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [18]:
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms,
                                                                               batch_size=32)

train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x7b482832b460>,
 ['Audemars Piguet',
  'Breguet',
  'Breitling',
  'Maurice Lacroix',
  'Omega',
  'Patek Philippe',
  'Rado',
  'Rolex',
  'Tissot',
  'Vacheron Constantin'])

In [19]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 53.9MB/s]


In [20]:

summary(model=model,
        input_size=(32, 3, 224, 224),

        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 1000]           --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

In [21]:
for param in model.features.parameters():
  param.requires_grad = False

In [22]:

torch.manual_seed(42)
torch.cuda.manual_seed(42)

output_shape = len(class_names)

model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=output_shape,
                    bias=True))

In [23]:

summary(model=model,
        input_size=(32, 3, 224, 224),

        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 10]             --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

In [24]:
loss_fn  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [26]:
torch.manual_seed(42)


from timeit import default_timer as timer
start_time = timer()

results = engine.train(model=model,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=25)

end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/25 [00:00<?, ?it/s]

RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

In [ ]:
try:
    from helper_functions import plot_loss_curves
except:
    print("[INFO] Couldn't find helper_functions.py, downloading...")
    with open("helper_functions.py", "wb") as f:
        import requests
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
        f.write(request.content)
    from helper_functions import plot_loss_curves

# Plot the loss curves of our model
plot_loss_curves(results)